# Stress tests
CIFAR-10 stress tests for the magazine article, each changing one setting of the main runs in setup_remote.ipynb; results go to a separate Results/stress folder.


Prepare the project and its imports, as in setup_remote.ipynb.


In [1]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
REPOSITORY_URL = "https://github.com/s4meer-git/BPDC_FL_Framework.git"

if ON_COLAB:
    PROJECT_ROOT = Path("/content/FLFrameWork")
    if (PROJECT_ROOT / ".git").exists():
        if "configuration" in sys.modules or "main" in sys.modules:
            raise RuntimeError("Restart the Python session before fetching code so old functions are not kept in memory.")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "JPNotebooks" / "requirements.txt")], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "JPNotebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "configuration.py").is_file():
    raise FileNotFoundError("Open this notebook from the project folder, or set PROJECT_ROOT to your checkout.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from configuration import configuration
from support_modules.experiment import prepare_runs, run_experiments, comparison_csvs
print("Project:", PROJECT_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Project: /content/FLFrameWork
GPU: NVIDIA L4


Stress-test results are saved as Results/stress/<test>/cifar10/RUN1, RUN2, RUN3/<selfish|non_selfish>/<method>.csv.


In [2]:
# SAFETY-FIRST DRIVE BACKUP
# Preserve the complete Results/stress/<test>/<dataset>/RUN<n>/<scenario>/method.csv structure.
# Back up and VERIFY each CSV after every completed round, and again at run completion.
from pathlib import Path
from google.colab import drive
import support_modules.experiment as experiment
import shutil
import hashlib
import os
import uuid

drive.mount("/content/drive")
RESULTS_ROOT = (PROJECT_ROOT / "Results" / "stress").resolve()
BACKUP_ROOT = Path("/content/drive/MyDrive/FLFrameWork_ab3/Results")
BASE_LOCAL_RESULTS = (PROJECT_ROOT / "Results").resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

# The repo's backup_path kept only 3 directory levels and could collide across ablations.
def safe_backup_path(folder, backup_root):
    if backup_root is None:
        return None
    relative_folder = Path(folder).resolve().relative_to(BASE_LOCAL_RESULTS)
    return Path(backup_root) / relative_folder

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_backup_csv(path, backup_folder):
    if backup_folder is None:
        raise RuntimeError("Google Drive backup folder is not configured")
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"Expected experiment CSV does not exist: {source}")
    target_dir = Path(backup_folder)
    target_dir.mkdir(parents=True, exist_ok=True)
    destination = target_dir / source.name
    if source.resolve() == destination.resolve():
        return
    partial = target_dir / (f".{source.name}.{uuid.uuid4().hex}.partial")
    try:
        shutil.copy2(source, partial)
        source_hash = _sha256(source)
        if _sha256(partial) != source_hash:
            raise IOError(f"Incomplete Google Drive upload: {destination}")
        os.replace(partial, destination)  # Replace only this method's CSV, never a folder.
        if _sha256(destination) != source_hash:
            raise IOError(f"Google Drive file mismatch: {destination}")
    finally:
        partial.unlink(missing_ok=True)
    print("[DRIVE VERIFIED]", destination)

# run_experiments and comparison_csvs use functions in this module's global namespace.
experiment.backup_path = safe_backup_path
experiment.backup_csv = safe_backup_csv
backup_path = safe_backup_path
backup_csv = safe_backup_csv

# Probe the exact target directory: refuse to train unless write/rename/read work.
probe = RESULTS_ROOT / ("_drive_probe_" + uuid.uuid4().hex + ".txt")
probe.write_text("drive backup verified", encoding="utf-8")
try:
    target_folder = backup_path(probe.parent, BACKUP_ROOT)
    backup_csv(probe, target_folder)
    target = target_folder / probe.name
    assert target.read_text(encoding="utf-8") == probe.read_text(encoding="utf-8")
finally:
    probe.unlink(missing_ok=True)
    if 'target' in locals():
        target.unlink(missing_ok=True)

print("\nBackup enabled:", BACKUP_ROOT / "stress")
print("Backups checked after EVERY completed communication round and run finish.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[DRIVE VERIFIED] /content/drive/MyDrive/FLFrameWork_ab3/Results/stress/_drive_probe_106b3d95e8a64c4abefe9c807e70cc9b.txt

Backup enabled: /content/drive/MyDrive/FLFrameWork_ab3/Results/stress
Backups checked after EVERY completed communication round and run finish.


Choose the tests and seeds; ONLY_SEEDS limits this session to some seeds while keeping RUN numbering, so several sessions can share the work.


In [ ]:
import copy

config = configuration()

DATASET = "tinyimagenet"
METHODS = ["fedavg", "fcfl", "fedcda", "fedmut", "fairrfl"]
SEEDS = [42, 43, 44]  # FIXED seed order: RUN1=42, RUN2=43, RUN3=44
ONLY_SEEDS = [44]   # Can use [43], [44], [42, 43], or None for all
RERUN_COMPLETED = False
HONEST_ONLY = True   # Only non_selfish runs

config.num_of_clients = 30
config.rounds = 40
config.local_epochs = 5
config.batch_size = 64
config.lr = 0.01
config.dirichlet_alpha = 0.5
MOMENTUM = {"mnist": 0.5, "cifar10": 0.9, "tinyimagenet": 0.9}
config.selfish_fraction = 0.05
config.selfish_strength = 0.7
config.selfish_start_round = 3
config.fairrfl_q = 1.0
config.fairrfl_tau = 2.5
config.fairrfl_dynamic_q = True
config.fairrfl_lipschitz = None
config.fedcda_warmup = 10
config.fcfl_alpha = 0.5
config.fcfl_fraction = 1.0

TESTS = {
    "quiet_attacker": {"methods": METHODS, "selfish": [True], "settings": {"selfish_strength": 0.05}},
    "fairrfl_screen_only": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_q": 0.0}},
    "fairrfl_weighting_only": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_tau": 1e9}},
    "fairrfl_screen_before_weighting": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_screen_before_weighting": True}},
    "fairrfl_screen_sqrt_steps": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_screen_before_weighting": True, "fairrfl_screen_sqrt_steps": True}},
    "fedavg_norm_clip": {"methods": ["fedavg"], "selfish": [False, True], "settings": {"fedavg_norm_clip": True}},
    "nine_attackers": {"methods": METHODS, "selfish": [True], "settings": {"selfish_fraction": 0.3}},
    "skew_alpha_0.1": {"methods": METHODS, "selfish": [False, True], "settings": {"dirichlet_alpha": 0.1}},
}
SELECTED_TESTS = [
    "fairrfl_screen_only",
    "fairrfl_weighting_only",
    "fairrfl_screen_before_weighting",
]

# SELECTED_TESTS = [
#     "fairrfl_screen_sqrt_steps",
#     "fedavg_norm_clip",
# ]

runs = []
for name in SELECTED_TESTS:
    test = TESTS[name]
    for selfish in test["selfish"]:
        if HONEST_ONLY and selfish:
            continue
        test_config = copy.deepcopy(config)
        test_config.selfishness = selfish
        for setting, value in test["settings"].items():
            setattr(test_config, setting, value)
        # Always pass ALL seeds to preserve RUN numbers, then filter.
        planned = prepare_runs(test_config, [DATASET], test["methods"], RESULTS_ROOT / name, MOMENTUM, SEEDS)
        runs += [r for r in planned if ONLY_SEEDS is None or r[0].seed in ONLY_SEEDS]

assert runs, "No experiments selected. Check ONLY_SEEDS, HONEST_ONLY and SELECTED_TESTS."
planned_backups = []
for run_config, folder in runs:
    destination = backup_path(folder, BACKUP_ROOT) / f"{run_config.method}.csv"
    planned_backups.append(destination)
    print(f"seed={run_config.seed}: {destination}")
assert len(planned_backups) == len(set(planned_backups)), "Two runs would write to the same Drive CSV!"
print("Total scheduled runs:", len(runs))

seed=43: /content/drive/MyDrive/FLFrameWork_ab2/Results/stress/fairrfl_screen_only/tinyimagenet/RUN2/non_selfish/fairrfl.csv
seed=43: /content/drive/MyDrive/FLFrameWork_ab2/Results/stress/fairrfl_weighting_only/tinyimagenet/RUN2/non_selfish/fairrfl.csv
seed=43: /content/drive/MyDrive/FLFrameWork_ab2/Results/stress/fairrfl_screen_before_weighting/tinyimagenet/RUN2/non_selfish/fairrfl.csv
seed=43: /content/drive/MyDrive/FLFrameWork_ab2/Results/stress/fairrfl_screen_sqrt_steps/tinyimagenet/RUN2/non_selfish/fairrfl.csv
seed=43: /content/drive/MyDrive/FLFrameWork_ab2/Results/stress/fedavg_norm_clip/tinyimagenet/RUN2/non_selfish/fedavg.csv
Total scheduled runs: 5


Run the selected tests; completed runs are skipped, diverged runs are recorded, and incomplete runs restart from round 1.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running training")

# Each run is copied after EVERY round, and its final results immediately on completion.
# Reconnect with the same settings to skip completed CSVs already on Drive.
run_summary = run_experiments(runs, backup_root=BACKUP_ROOT, rerun_completed=RERUN_COMPLETED)
display(run_summary)

100%|██████████| 248M/248M [00:37<00:00, 6.55MB/s]


Streaming output truncated to the last 5000 lines.
Local epoch 5 started
Local epoch 5 completed batches: 52 average loss: 3.2258
Training client 4
Local epoch 1 started
Local epoch 1 completed batches: 56 average loss: 4.2087
Local epoch 2 started
Local epoch 2 completed batches: 56 average loss: 3.8591
Local epoch 3 started
Local epoch 3 completed batches: 56 average loss: 3.6333
Local epoch 4 started
Local epoch 4 completed batches: 56 average loss: 3.4613
Local epoch 5 started
Local epoch 5 completed batches: 56 average loss: 3.3229
Training client 5
Local epoch 1 started
Local epoch 1 completed batches: 56 average loss: 4.1771
Local epoch 2 started
Local epoch 2 completed batches: 56 average loss: 3.7875
Local epoch 3 started
Local epoch 3 completed batches: 56 average loss: 3.5899
Local epoch 4 started
Local epoch 4 completed batches: 56 average loss: 3.4245
Local epoch 5 started
Local epoch 5 completed batches: 56 average loss: 3.2651
Training client 6
Local epoch 1 started
Loca

Run this after every seed has finished to save mean ± sample standard deviation for each test and scenario.


In [ ]:
from analysis.results import seed_summary
from analysis.plots import format_seed_summary
from support_modules.experiment import backup_csv, backup_path


# One summary per test and scenario, saved as summary_<scenario>.csv in that test's dataset folder.
comparisons = {}
for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    comparisons.setdefault((folder.parent.parent, folder.name), {})[folder.parent.name] = csv_data

for (dataset_folder, scenario), csv_by_seed in comparisons.items():
    means, deviations, per_seed, partial = seed_summary(csv_by_seed)
    summary = format_seed_summary(means, deviations)
    summary_path = dataset_folder / f"summary_{scenario}.csv"
    summary.to_csv(summary_path)
    per_seed.to_csv(dataset_folder / f"summary_{scenario}_per_seed.csv", index=False)
    print(dataset_folder.relative_to(RESULTS_ROOT), scenario, "| RUNs:", sorted(csv_by_seed))
    for method, missing in partial.items():
        print(f"{method} diverged for seeds {missing}; it is left out of the mean ± std table.")
    display(summary)
    backup_csv(summary_path, backup_path(dataset_folder, BACKUP_ROOT))
